# 01 — Proseg cell QC and ResolVI-ready filtering

This notebook prepares each Proseg cell-level Visium HD object **before** ResolVI.
It keeps the original count matrix unnormalized, annotates mitochondrial,
ribosomal, hemoglobin, and deprecated features, writes an unfiltered QC-annotated
copy, and then writes a filtered copy for sample-level ResolVI.

Default cell filters follow the requested rules:

- `mt_high = pct_counts_mt >= 20`
- remove cells with `mt_high`
- remove cells with fewer than 16 detected nondeprecated genes
- after cell filtering, retain every nondeprecated gene detected in at least one retained cell

The colon-cancer pair is included by default but can be disabled in the configuration.

## Cell-cycle scoring

After cell and gene QC, the notebook scores S phase and G2/M phase on a
**temporary normalized/log-transformed copy** of the QC-passed cells. Only
`S_score`, `G2M_score`, and `phase` are copied back. The filtered object's
`X` remains the original sparse integer count matrix required by ResolVI.

`CELL_CYCLE_BACKEND = "auto"` uses RAPIDS SingleCell on the selected GPU for
large samples when available and otherwise falls back to Scanpy. One GPU is
used per notebook process; the four GPUs are most useful for running disjoint
sample shards in separate kernels.


In [1]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import time
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

SAMPLE_INFO = {
    "Screen_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen"
    },
    "C2D15_39_21": {
        "patient": "patient_39_21",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15"
    },
    "Screen_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "Screen"
    },
    "C2D15_17_26": {
        "patient": "patient_17_26",
        "cancer_type": "NSCLC",
        "biopsy_stage": "C2D15"
    },
    "Screen_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_18_23": {
        "patient": "patient_18_23",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_16_22": {
        "patient": "patient_16_22",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "Screen"
    },
    "C2D15_30_16": {
        "patient": "patient_30_16",
        "cancer_type": "melanoma",
        "biopsy_stage": "C2D15"
    },
    "Screen_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "Screen"
    },
    "C2D15_23_25": {
        "patient": "patient_23_25",
        "cancer_type": "colon_cancer",
        "biopsy_stage": "C2D15"
    }
}
DEFAULT_SIGNATURES = {
    "immune_core": [
        "PTPRC",
        "CD3D",
        "CD3E",
        "TRAC",
        "LCK",
        "IL32",
        "LST1",
        "TYROBP",
        "FCER1G",
        "MS4A1",
        "CD79A",
        "NKG7",
        "KLRD1",
        "AIF1",
        "CTSS"
    ],
    "T_cell": [
        "CD3D",
        "CD3E",
        "TRAC",
        "LCK",
        "CD247",
        "IL32",
        "LTB"
    ],
    "CD4_helper": [
        "IL7R",
        "LTB",
        "CCR7",
        "MAL",
        "TCF7",
        "LEF1"
    ],
    "CD8_T": [
        "CD8A",
        "CD8B",
        "CTSW",
        "CCL5",
        "TRAC",
        "LCK"
    ],
    "Treg": [
        "FOXP3",
        "IL2RA",
        "CTLA4",
        "TIGIT",
        "IKZF2",
        "TNFRSF18"
    ],
    "NK": [
        "KLRD1",
        "NKG7",
        "GNLY",
        "PRF1",
        "FCER1G",
        "TYROBP",
        "TRDC"
    ],
    "B_cell": [
        "MS4A1",
        "CD79A",
        "CD79B",
        "CD37",
        "CD74",
        "HLA-DRA",
        "CD22"
    ],
    "plasma_cell": [
        "MZB1",
        "JCHAIN",
        "SDC1",
        "XBP1",
        "DERL3",
        "IGKC"
    ],
    "myeloid": [
        "LST1",
        "TYROBP",
        "FCER1G",
        "AIF1",
        "CTSS",
        "LILRB1",
        "CSTA",
        "SAT1"
    ],
    "mast_cell": [
        "TPSAB1",
        "TPSB2",
        "KIT",
        "CPA3",
        "MS4A2",
        "HDC"
    ],
    "endothelial": [
        "PECAM1",
        "VWF",
        "KDR",
        "ESAM",
        "ENG",
        "EMCN",
        "RAMP2",
        "RGCC",
        "PLVAP",
        "CA4"
    ],
    "epithelial_keratin": [
        "EPCAM",
        "TACSTD2",
        "KRT7",
        "KRT8",
        "KRT18",
        "KRT19",
        "MUC1",
        "KRT5",
        "KRT6A",
        "KRT6B",
        "KRT14",
        "KRT17",
        "TP63",
        "SFN",
        "DSG3",
        "CEACAM5",
        "CEACAM6",
        "MSLN",
        "KRT20",
        "MUC13",
        "TFF3"
    ],
    "keratinocyte": [
        "DSP",
        "DMKN",
        "KRT10",
        "DSG1",
        "DSC3",
        "IVL",
        "SFN",
        "KRTDAP"
    ],
    "melanoma_melanocytic": [
        "MLANA",
        "PMEL",
        "TYR",
        "DCT",
        "MITF",
        "SOX10",
        "S100B"
    ],
    "melanoma_dedifferentiated": [
        "AXL",
        "NGFR",
        "SOX9"
    ],
    "fibroblast": [
        "COL1A1",
        "COL1A2",
        "COL3A1",
        "DCN",
        "LUM",
        "PDGFRA",
        "COL6A1",
        "COL6A2",
        "C7"
    ],
    "mural": [
        "RGS5",
        "CSPG4",
        "MCAM",
        "PDGFRB",
        "ACTA2",
        "TAGLN",
        "MYL9",
        "DES"
    ],
    "erythroid": [
        "HBA1",
        "HBA2",
        "HBB",
        "HBD",
        "ALAS2",
        "AHSP",
        "GYPA"
    ]
}

print('Imports loaded.')


Imports loaded.


In [2]:
# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
PROJECT_ROOT = Path('/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057')
TMP_ROOT = PROJECT_ROOT / 'tmp'

# Add any additional Proseg output roots here. Exact sample-specific matches
# are preferred; recursive fallback matches must be unambiguous.
PROSEG_INPUT_ROOTS = [
    TMP_ROOT / 'proseg_shared_qcprior_v1',
    TMP_ROOT / 'proseg_shared_qcprior_all_v1',
    TMP_ROOT / 'proseg_output_bigTiff',
]
EXPLICIT_INPUT_PATHS = {
    # 'Screen_30_16': Path('/custom/path/Screen_30_16_proseg_cdata.h5ad'),
}

PIPELINE_ROOT = TMP_ROOT / 'proseg_resolvi_immune_enrichment_v1'
CONFIG_ROOT = PIPELINE_ROOT / '00_config'
QC_ANNOTATED_ROOT = PIPELINE_ROOT / '01_qc_annotated'
QC_FILTERED_ROOT = PIPELINE_ROOT / '01_qc_filtered'
QC_REPORT_ROOT = PIPELINE_ROOT / '01_qc_reports'

for path in (CONFIG_ROOT, QC_ANNOTATED_ROOT, QC_FILTERED_ROOT, QC_REPORT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

INCLUDE_COLON = True
SECTION_NAMES = [
    sample
    for sample, meta in SAMPLE_INFO.items()
    if INCLUDE_COLON or meta['cancer_type'] != 'colon_cancer'
]

MT_HIGH_PERCENT = 20.0
MIN_GENES_PER_CELL = 16
MIN_CELLS_PER_GENE = 1
REMOVE_DEPRECATED_FROM_FILTERED = True
USE_NODEP_QC_FOR_CELL_FILTERS = True

H5AD_COMPRESSION = 'lzf'
USE_EXISTING_OUTPUTS = True
OVERWRITE_OUTPUTS = False
CONTINUE_ON_ERROR = True
PLOT_MAX_CELLS = 200_000
PLOT_DPI = 300
RANDOM_SEED = 0

PIPELINE_VERSION = "2026-07-28-cellcycle-v2"

# Cell-cycle scoring is calculated only for QC-passed cells.
RUN_CELL_CYCLE_SCORING = True

# "auto", "rapids", or "scanpy".
# auto uses RAPIDS for sufficiently large samples when the GPU stack imports.
CELL_CYCLE_BACKEND = "auto"
CELL_CYCLE_GPU_ID = 0
CELL_CYCLE_GPU_MIN_CELLS = 50_000
CELL_CYCLE_GPU_FALLBACK_TO_CPU = True

# The temporary scoring matrix is normalized and log1p-transformed.
# The ResolVI-ready raw integer count matrix is never modified.
CELL_CYCLE_TARGET_SUM = 1e4
CELL_CYCLE_CTRL_AS_REF = True
CELL_CYCLE_MIN_GENES_PER_PHASE = 3
CELL_CYCLE_RANDOM_STATE = RANDOM_SEED

S_PHASE_GENES = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM7", "MCM4", "RRM1", "UNG",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "UHRF1", "CENPU",
    "HELLS", "RFC2", "POLR1B", "NASP", "RAD51AP1", "GMNN", "WDR76",
    "SLBP", "CCNE2", "UBR7",
]

G2M_PHASE_GENES = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A",
    "NDC80", "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF",
    "TACC3", "FAM64A", "SMC4", "CCNB2", "CKAP2L", "CKAP2", "AURKB",
    "BUB1", "KIF11", "ANP32E", "TUBB4B",
]

# Exact human hemoglobin genes; pseudogenes are intentionally excluded.
HEMOGLOBIN_GENES = {
    'HBA1', 'HBA2', 'HBB', 'HBD', 'HBE1', 'HBG1', 'HBG2', 'HBM', 'HBQ1', 'HBZ'
}

print('Samples:', SECTION_NAMES)
print('Pipeline root:', PIPELINE_ROOT)


Samples: ['Screen_39_21', 'C2D15_39_21', 'Screen_17_26', 'C2D15_17_26', 'Screen_18_23', 'C2D15_18_23', 'Screen_16_22', 'C2D15_16_22', 'Screen_30_16', 'C2D15_30_16', 'Screen_23_25', 'C2D15_23_25']
Pipeline root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1


In [3]:
# ---------------------------------------------------------------------
# Path and feature helpers
# ---------------------------------------------------------------------
FEATURE_NAME_COLUMNS = [
    'gene_ids', 'gene_id', 'feature_ids', 'feature_id', 'id',
    'feature_name', 'gene', 'name', 'gene_name', 'gene_symbol', 'symbol',
]


def sample_paths(sample: str) -> dict[str, Path]:
    annotated_dir = QC_ANNOTATED_ROOT / sample
    filtered_dir = QC_FILTERED_ROOT / sample
    report_dir = QC_REPORT_ROOT / sample
    for directory in (annotated_dir, filtered_dir, report_dir):
        directory.mkdir(parents=True, exist_ok=True)
    return {
        'annotated_dir': annotated_dir,
        'filtered_dir': filtered_dir,
        'report_dir': report_dir,
        'annotated': annotated_dir / f'{sample}_proseg_qc_annotated.h5ad',
        'filtered': filtered_dir / f'{sample}_proseg_qc_filtered.h5ad',
        'summary': report_dir / f'{sample}_qc_summary.json',
    }


def locate_proseg_cdata(sample: str) -> Path:
    if sample in EXPLICIT_INPUT_PATHS:
        path = Path(EXPLICIT_INPUT_PATHS[sample])
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    exact = []
    for root in PROSEG_INPUT_ROOTS:
        exact.extend([
            root / sample / f'{sample}_proseg_sharedprior_cdata.h5ad',
            root / sample / f'{sample}_proseg_cdata.h5ad',
            root / f'{sample}_proseg_sharedprior_cdata.h5ad',
        ])
    exact = [path for path in exact if path.exists()]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        raise RuntimeError(
            f'Multiple exact Proseg inputs for {sample}: {[str(x) for x in exact]}'
        )

    fallback = []
    for root in PROSEG_INPUT_ROOTS:
        if root.exists():
            fallback.extend(root.rglob(f'{sample}*proseg*cdata*.h5ad'))
    fallback = sorted(set(path.resolve() for path in fallback))
    if len(fallback) == 1:
        return fallback[0]
    if not fallback:
        raise FileNotFoundError(
            f'No Proseg cdata found for {sample}. Checked roots: '
            f'{[str(x) for x in PROSEG_INPUT_ROOTS]}'
        )
    raise RuntimeError(
        f'Ambiguous Proseg inputs for {sample}: {[str(x) for x in fallback]}'
    )


def atomic_write_h5ad(adata: ad.AnnData, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.stem + '.tmp.h5ad')
    if temp.exists():
        temp.unlink()
    adata.write_h5ad(temp, compression=H5AD_COMPRESSION)
    temp.replace(path)


def write_json(payload, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    temp.replace(path)


def feature_text_arrays(var: pd.DataFrame) -> list[np.ndarray]:
    arrays = [np.asarray(var.index.astype(str), dtype=object)]
    for column in FEATURE_NAME_COLUMNS:
        if column in var.columns:
            arrays.append(np.asarray(var[column].astype(str), dtype=object))
    return arrays


def any_feature_match(var: pd.DataFrame, predicate) -> np.ndarray:
    mask = np.zeros(len(var), dtype=bool)
    for values in feature_text_arrays(var):
        upper = np.char.upper(values.astype(str))
        mask |= predicate(upper)
    return mask


def annotate_feature_classes(adata: ad.AnnData) -> None:
    adata.var['mt'] = any_feature_match(
        adata.var, lambda x: np.char.startswith(x, 'MT-') | np.char.startswith(x, 'MT_')
    )
    adata.var['ribo'] = any_feature_match(
        adata.var, lambda x: np.char.startswith(x, 'RPS') | np.char.startswith(x, 'RPL')
    )
    adata.var['hb'] = any_feature_match(
        adata.var, lambda x: np.isin(x, np.asarray(sorted(HEMOGLOBIN_GENES), dtype=str))
    )
    adata.var['is_deprecated'] = any_feature_match(
        adata.var, lambda x: np.char.startswith(x, 'DEPRECATED_')
    )


def ensure_integer_csr(adata: ad.AnnData, tolerance: float = 1e-6) -> float:
    X = sp.csr_matrix(adata.X)
    X.sum_duplicates()
    X.eliminate_zeros()
    if X.data.size:
        maximum_fractional_error = float(np.max(np.abs(X.data - np.rint(X.data))))
    else:
        maximum_fractional_error = 0.0
    if maximum_fractional_error > tolerance:
        raise ValueError(
            'Input Proseg matrix is not integer-like. Maximum fractional error: '
            f'{maximum_fractional_error}'
        )
    X.data = np.rint(X.data).astype(np.int32, copy=False)
    adata.X = X
    return maximum_fractional_error


def csr_row_sums(X) -> np.ndarray:
    return np.asarray(X.sum(axis=1)).ravel().astype(np.float64, copy=False)


def csr_col_sums(X) -> np.ndarray:
    return np.asarray(X.sum(axis=0)).ravel().astype(np.float64, copy=False)


def subset_row_sums(X: sp.csr_matrix, mask: np.ndarray) -> np.ndarray:
    if not np.any(mask):
        return np.zeros(X.shape[0], dtype=np.float64)
    return np.asarray(X[:, mask].sum(axis=1)).ravel().astype(np.float64, copy=False)


def subset_row_nnz(X: sp.csr_matrix, mask: np.ndarray) -> np.ndarray:
    if not np.any(mask):
        return np.zeros(X.shape[0], dtype=np.int32)
    return np.asarray(X[:, mask].getnnz(axis=1)).ravel().astype(np.int32, copy=False)


In [4]:
# ---------------------------------------------------------------------
# QC calculations and figures
# ---------------------------------------------------------------------
def calculate_qc_annotations(adata: ad.AnnData) -> None:
    X = sp.csr_matrix(adata.X)
    X.sum_duplicates()
    X.eliminate_zeros()
    adata.X = X

    total_counts = csr_row_sums(X)
    n_genes = np.diff(X.indptr).astype(np.int32, copy=False)

    mt_mask = adata.var['mt'].to_numpy(dtype=bool)
    ribo_mask = adata.var['ribo'].to_numpy(dtype=bool)
    hb_mask = adata.var['hb'].to_numpy(dtype=bool)
    dep_mask = adata.var['is_deprecated'].to_numpy(dtype=bool)

    mt_counts = subset_row_sums(X, mt_mask)
    ribo_counts = subset_row_sums(X, ribo_mask)
    hb_counts = subset_row_sums(X, hb_mask)
    dep_counts = subset_row_sums(X, dep_mask)
    dep_n_genes = subset_row_nnz(X, dep_mask)

    denominator = np.maximum(total_counts, 1.0)
    adata.obs['total_counts'] = total_counts
    adata.obs['n_genes_by_counts'] = n_genes
    adata.obs['total_counts_mt'] = mt_counts
    adata.obs['pct_counts_mt'] = 100.0 * mt_counts / denominator
    adata.obs['total_counts_ribo'] = ribo_counts
    adata.obs['pct_counts_ribo'] = 100.0 * ribo_counts / denominator
    adata.obs['total_counts_hb'] = hb_counts
    adata.obs['pct_counts_hb'] = 100.0 * hb_counts / denominator
    adata.obs['deprecated_counts'] = dep_counts
    adata.obs['deprecated_pct'] = 100.0 * dep_counts / denominator

    nodep_total = np.clip(total_counts - dep_counts, 0.0, None)
    nodep_n_genes = np.maximum(n_genes - dep_n_genes, 0)
    nodep_mt = subset_row_sums(X, mt_mask & ~dep_mask)
    nodep_ribo = subset_row_sums(X, ribo_mask & ~dep_mask)
    nodep_hb = subset_row_sums(X, hb_mask & ~dep_mask)
    nodep_denominator = np.maximum(nodep_total, 1.0)

    adata.obs['nodep_total_counts'] = nodep_total
    adata.obs['nodep_n_genes_by_counts'] = nodep_n_genes
    adata.obs['nodep_total_counts_mt'] = nodep_mt
    adata.obs['nodep_pct_counts_mt'] = 100.0 * nodep_mt / nodep_denominator
    adata.obs['nodep_total_counts_ribo'] = nodep_ribo
    adata.obs['nodep_pct_counts_ribo'] = 100.0 * nodep_ribo / nodep_denominator
    adata.obs['nodep_total_counts_hb'] = nodep_hb
    adata.obs['nodep_pct_counts_hb'] = 100.0 * nodep_hb / nodep_denominator

    adata.var['total_counts'] = csr_col_sums(X)
    adata.var['n_cells_by_counts'] = np.bincount(X.indices, minlength=X.shape[1]).astype(np.int64)


def add_cell_filter_flags(adata: ad.AnnData) -> tuple[str, str]:
    if USE_NODEP_QC_FOR_CELL_FILTERS:
        mt_column = 'nodep_pct_counts_mt'
        genes_column = 'nodep_n_genes_by_counts'
    else:
        mt_column = 'pct_counts_mt'
        genes_column = 'n_genes_by_counts'

    mt_high = adata.obs[mt_column].to_numpy(dtype=float) >= MT_HIGH_PERCENT
    low_genes = adata.obs[genes_column].to_numpy(dtype=int) < MIN_GENES_PER_CELL
    adata.obs['mt_high'] = mt_high
    adata.obs['low_genes'] = low_genes
    adata.obs['qc_pass'] = ~(mt_high | low_genes)
    adata.obs['qc_filter_reason'] = pd.Categorical(
        np.select(
            [mt_high & low_genes, mt_high, low_genes],
            ['mt_high+low_genes', 'mt_high', 'low_genes'],
            default='pass',
        ),
        categories=['pass', 'mt_high', 'low_genes', 'mt_high+low_genes'],
    )
    return mt_column, genes_column



def make_cell_cycle_scoring_adata(filtered: ad.AnnData) -> ad.AnnData:
    """Create a lean temporary AnnData containing float32 counts only."""
    X = sp.csr_matrix(filtered.X, dtype=np.float32)
    X.sum_duplicates()
    X.eliminate_zeros()

    return ad.AnnData(
        X=X,
        obs=pd.DataFrame(index=filtered.obs_names.copy()),
        var=pd.DataFrame(index=filtered.var_names.copy()),
    )


def available_cell_cycle_genes(
    var_names: pd.Index,
) -> tuple[list[str], list[str], list[str], list[str]]:
    available = set(var_names.astype(str))
    s_present = [gene for gene in S_PHASE_GENES if gene in available]
    g2m_present = [gene for gene in G2M_PHASE_GENES if gene in available]
    s_missing = [gene for gene in S_PHASE_GENES if gene not in available]
    g2m_missing = [gene for gene in G2M_PHASE_GENES if gene not in available]
    return s_present, g2m_present, s_missing, g2m_missing


def rapids_cell_cycle_is_available(gpu_id: int) -> tuple[bool, str]:
    """Check imports and the requested visible GPU without changing the object."""
    try:
        import cupy as cp
        import rapids_singlecell as rsc  # noqa: F401

        n_devices = int(cp.cuda.runtime.getDeviceCount())
        if n_devices <= int(gpu_id):
            return (
                False,
                f"Requested visible GPU {gpu_id}, but CuPy sees {n_devices} device(s).",
            )
        return True, f"CuPy sees {n_devices} visible GPU device(s)."
    except Exception as exc:
        return False, f"{type(exc).__name__}: {exc}"


def run_cell_cycle_scanpy(
    scoring_adata: ad.AnnData,
    s_genes: list[str],
    g2m_genes: list[str],
) -> None:
    """Normalize, log1p-transform, and score on CPU."""
    sc.pp.normalize_total(
        scoring_adata,
        target_sum=float(CELL_CYCLE_TARGET_SUM),
    )
    sc.pp.log1p(scoring_adata)
    sc.tl.score_genes_cell_cycle(
        scoring_adata,
        s_genes=s_genes,
        g2m_genes=g2m_genes,
        use_raw=False,
        random_state=int(CELL_CYCLE_RANDOM_STATE),
        ctrl_as_ref=bool(CELL_CYCLE_CTRL_AS_REF),
    )


def run_cell_cycle_rapids(
    scoring_adata: ad.AnnData,
    s_genes: list[str],
    g2m_genes: list[str],
    gpu_id: int,
) -> None:
    """Normalize, log1p-transform, and score on one selected GPU."""
    import cupy as cp
    import rapids_singlecell as rsc

    with cp.cuda.Device(int(gpu_id)):
        rsc.get.anndata_to_GPU(
            scoring_adata,
            convert_all=False,
        )
        rsc.pp.normalize_total(
            scoring_adata,
            target_sum=float(CELL_CYCLE_TARGET_SUM),
        )
        rsc.pp.log1p(scoring_adata)
        rsc.tl.score_genes_cell_cycle(
            scoring_adata,
            s_genes=s_genes,
            g2m_genes=g2m_genes,
            use_raw=False,
            random_state=int(CELL_CYCLE_RANDOM_STATE),
            ctrl_as_ref=bool(CELL_CYCLE_CTRL_AS_REF),
        )

        # Only X is GPU-backed in this lean temporary object. Moving it back
        # makes cleanup deterministic and ensures no CuPy matrix escapes.
        rsc.get.anndata_to_CPU(
            scoring_adata,
            convert_all=False,
        )
        cp.cuda.Stream.null.synchronize()
        cp.get_default_memory_pool().free_all_blocks()


def score_cell_cycle_on_qc_passed(
    filtered: ad.AnnData,
) -> dict:
    """
    Score cell cycle on QC-passed cells without changing filtered.X.

    Returns a JSON-serializable diagnostics dictionary and adds:
      filtered.obs["S_score"]
      filtered.obs["G2M_score"]
      filtered.obs["phase"]
      filtered.obs["cell_cycle_scored"]
    """
    diagnostics = {
        "enabled": bool(RUN_CELL_CYCLE_SCORING),
        "backend_requested": str(CELL_CYCLE_BACKEND),
        "backend_used": None,
        "gpu_id": int(CELL_CYCLE_GPU_ID),
        "target_sum": float(CELL_CYCLE_TARGET_SUM),
        "ctrl_as_ref": bool(CELL_CYCLE_CTRL_AS_REF),
        "n_cells_scored": 0,
        "s_genes_present": [],
        "g2m_genes_present": [],
        "s_genes_missing": [],
        "g2m_genes_missing": [],
        "status": "disabled",
        "gpu_availability_message": None,
        "gpu_error": None,
    }

    filtered.obs["S_score"] = np.nan
    filtered.obs["G2M_score"] = np.nan
    filtered.obs["phase"] = pd.Categorical(
        ["not_scored"] * filtered.n_obs,
        categories=["G1", "S", "G2M", "not_scored"],
    )
    filtered.obs["cell_cycle_scored"] = False

    if not RUN_CELL_CYCLE_SCORING:
        return diagnostics

    if filtered.n_obs == 0:
        diagnostics["status"] = "no_qc_passed_cells"
        return diagnostics

    (
        s_present,
        g2m_present,
        s_missing,
        g2m_missing,
    ) = available_cell_cycle_genes(filtered.var_names)

    diagnostics["s_genes_present"] = s_present
    diagnostics["g2m_genes_present"] = g2m_present
    diagnostics["s_genes_missing"] = s_missing
    diagnostics["g2m_genes_missing"] = g2m_missing

    if (
        len(s_present) < int(CELL_CYCLE_MIN_GENES_PER_PHASE)
        or len(g2m_present) < int(CELL_CYCLE_MIN_GENES_PER_PHASE)
    ):
        diagnostics["status"] = "insufficient_cell_cycle_genes"
        print(
            "[warning] Cell-cycle scoring skipped: "
            f"{len(s_present)} S genes and {len(g2m_present)} G2M genes "
            f"were present; minimum per phase is "
            f"{CELL_CYCLE_MIN_GENES_PER_PHASE}."
        )
        return diagnostics

    requested = str(CELL_CYCLE_BACKEND).lower()
    if requested not in {"auto", "rapids", "scanpy"}:
        raise ValueError(
            "CELL_CYCLE_BACKEND must be 'auto', 'rapids', or 'scanpy'; "
            f"observed {CELL_CYCLE_BACKEND!r}."
        )

    rapids_ok, rapids_message = rapids_cell_cycle_is_available(
        int(CELL_CYCLE_GPU_ID)
    )
    diagnostics["gpu_availability_message"] = rapids_message

    use_rapids = (
        requested == "rapids"
        or (
            requested == "auto"
            and filtered.n_obs >= int(CELL_CYCLE_GPU_MIN_CELLS)
            and rapids_ok
        )
    )

    if requested == "rapids" and not rapids_ok:
        if not CELL_CYCLE_GPU_FALLBACK_TO_CPU:
            raise RuntimeError(
                "RAPIDS cell-cycle scoring was requested but unavailable: "
                + rapids_message
            )
        print(
            "[warning] RAPIDS requested but unavailable; "
            "falling back to Scanpy:",
            rapids_message,
        )
        use_rapids = False

    scoring_adata = make_cell_cycle_scoring_adata(filtered)
    started = time.time()

    if use_rapids:
        try:
            print(
                "Cell-cycle scoring backend: RAPIDS SingleCell "
                f"(visible GPU {CELL_CYCLE_GPU_ID})"
            )
            run_cell_cycle_rapids(
                scoring_adata,
                s_present,
                g2m_present,
                int(CELL_CYCLE_GPU_ID),
            )
            diagnostics["backend_used"] = "rapids_singlecell"
        except Exception as exc:
            diagnostics["gpu_error"] = f"{type(exc).__name__}: {exc}"
            if not CELL_CYCLE_GPU_FALLBACK_TO_CPU:
                raise

            print(
                "[warning] RAPIDS cell-cycle scoring failed; "
                "restarting the temporary scoring object on CPU:",
                diagnostics["gpu_error"],
            )
            del scoring_adata
            gc.collect()
            scoring_adata = make_cell_cycle_scoring_adata(filtered)
            run_cell_cycle_scanpy(
                scoring_adata,
                s_present,
                g2m_present,
            )
            diagnostics["backend_used"] = "scanpy_cpu_fallback"
    else:
        print("Cell-cycle scoring backend: Scanpy CPU")
        run_cell_cycle_scanpy(
            scoring_adata,
            s_present,
            g2m_present,
        )
        diagnostics["backend_used"] = "scanpy_cpu"

    s_score = scoring_adata.obs["S_score"].to_numpy(dtype=np.float64)
    g2m_score = scoring_adata.obs["G2M_score"].to_numpy(dtype=np.float64)
    phase = scoring_adata.obs["phase"].astype(str).to_numpy()

    if not np.isfinite(s_score).all() or not np.isfinite(g2m_score).all():
        raise FloatingPointError(
            "Cell-cycle scoring produced non-finite S_score or G2M_score."
        )

    filtered.obs["S_score"] = s_score
    filtered.obs["G2M_score"] = g2m_score
    filtered.obs["phase"] = pd.Categorical(
        phase,
        categories=["G1", "S", "G2M", "not_scored"],
    )
    filtered.obs["cell_cycle_scored"] = True

    diagnostics["n_cells_scored"] = int(filtered.n_obs)
    diagnostics["status"] = "completed"
    diagnostics["runtime_seconds"] = float(time.time() - started)
    diagnostics["phase_counts"] = {
        str(key): int(value)
        for key, value in filtered.obs["phase"].value_counts().items()
    }

    del scoring_adata
    gc.collect()
    return diagnostics


def propagate_cell_cycle_to_annotated(
    annotated: ad.AnnData,
    filtered: ad.AnnData,
) -> None:
    """Copy scores to the full QC-annotated object; failed cells remain unscored."""
    annotated.obs["S_score"] = np.nan
    annotated.obs["G2M_score"] = np.nan
    annotated.obs["cell_cycle_scored"] = False

    phase = np.full(annotated.n_obs, "not_scored", dtype=object)
    positions = annotated.obs_names.get_indexer(filtered.obs_names)
    if np.any(positions < 0):
        raise KeyError(
            "At least one filtered cell could not be mapped back to "
            "the full QC-annotated object."
        )

    annotated.obs.iloc[
        positions,
        annotated.obs.columns.get_loc("S_score"),
    ] = filtered.obs["S_score"].to_numpy(dtype=np.float64)

    annotated.obs.iloc[
        positions,
        annotated.obs.columns.get_loc("G2M_score"),
    ] = filtered.obs["G2M_score"].to_numpy(dtype=np.float64)

    scored = filtered.obs["cell_cycle_scored"].to_numpy(dtype=bool)
    annotated.obs.iloc[
        positions,
        annotated.obs.columns.get_loc("cell_cycle_scored"),
    ] = scored

    filtered_phase = filtered.obs["phase"].astype(str).to_numpy()
    phase[positions] = filtered_phase
    annotated.obs["phase"] = pd.Categorical(
        phase,
        categories=["G1", "S", "G2M", "not_scored"],
    )


def save_cell_cycle_figures(
    filtered: ad.AnnData,
    sample: str,
    report_dir: Path,
) -> None:
    """Save phase counts and S-versus-G2M scores for scored QC-passed cells."""
    if (
        "cell_cycle_scored" not in filtered.obs
        or not filtered.obs["cell_cycle_scored"].any()
    ):
        return

    phase_counts = filtered.obs["phase"].value_counts().reindex(
        ["G1", "S", "G2M"],
        fill_value=0,
    )
    fig, ax = plt.subplots(figsize=(7, 5))
    phase_counts.plot(kind="bar", ax=ax)
    ax.set_xlabel("Cell-cycle phase")
    ax.set_ylabel("Number of QC-passed cells")
    ax.set_title(f"{sample}: cell-cycle phase")
    ax.tick_params(axis="x", rotation=0)
    fig.tight_layout()
    fig.savefig(
        report_dir / f"{sample}_cell_cycle_phase.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)

    rng = np.random.default_rng(RANDOM_SEED)
    if filtered.n_obs > PLOT_MAX_CELLS:
        idx = np.sort(
            rng.choice(
                filtered.n_obs,
                PLOT_MAX_CELLS,
                replace=False,
            )
        )
    else:
        idx = np.arange(filtered.n_obs)

    obs = filtered.obs.iloc[idx]
    fig, ax = plt.subplots(figsize=(7, 6))
    hb = ax.hexbin(
        obs["S_score"].to_numpy(dtype=float),
        obs["G2M_score"].to_numpy(dtype=float),
        gridsize=100,
        mincnt=1,
    )
    ax.set_xlabel("S_score")
    ax.set_ylabel("G2M_score")
    ax.set_title(f"{sample}: QC-passed cell-cycle scores")
    fig.colorbar(hb, ax=ax, label="Number of cells")
    fig.tight_layout()
    fig.savefig(
        report_dir / f"{sample}_cell_cycle_scores_hexbin.png",
        dpi=PLOT_DPI,
        bbox_inches="tight",
    )
    plt.close(fig)


def save_qc_figures(adata: ad.AnnData, sample: str, report_dir: Path) -> None:
    rng = np.random.default_rng(RANDOM_SEED)
    if adata.n_obs > PLOT_MAX_CELLS:
        idx = np.sort(rng.choice(adata.n_obs, PLOT_MAX_CELLS, replace=False))
    else:
        idx = np.arange(adata.n_obs)
    obs = adata.obs.iloc[idx]

    for column in ('nodep_n_genes_by_counts', 'nodep_pct_counts_mt', 'nodep_total_counts'):
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(obs[column].to_numpy(dtype=float), bins=100)
        ax.set_xlabel(column)
        ax.set_ylabel('Number of cells')
        ax.set_title(f'{sample}: {column}')
        fig.tight_layout()
        fig.savefig(report_dir / f'{sample}_{column}_hist.png', dpi=PLOT_DPI, bbox_inches='tight')
        plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 6))
    hb = ax.hexbin(
        obs['nodep_n_genes_by_counts'].to_numpy(dtype=float),
        obs['nodep_total_counts'].to_numpy(dtype=float),
        C=obs['nodep_pct_counts_mt'].to_numpy(dtype=float),
        gridsize=100,
        reduce_C_function=np.mean,
        mincnt=1,
    )
    ax.set_xlabel('nodep_n_genes_by_counts')
    ax.set_ylabel('nodep_total_counts')
    ax.set_title(f'{sample}: cell QC')
    fig.colorbar(hb, ax=ax, label='Mean nodep_pct_counts_mt')
    fig.tight_layout()
    fig.savefig(report_dir / f'{sample}_qc_hexbin.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)

    counts = adata.obs['qc_filter_reason'].value_counts().reindex(
        ['pass', 'mt_high', 'low_genes', 'mt_high+low_genes'], fill_value=0
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    counts.plot(kind='bar', ax=ax)
    ax.set_xlabel('QC category')
    ax.set_ylabel('Number of cells')
    ax.set_title(f'{sample}: QC outcome')
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    fig.savefig(report_dir / f'{sample}_qc_outcome.png', dpi=PLOT_DPI, bbox_inches='tight')
    plt.close(fig)


In [5]:
# ---------------------------------------------------------------------
# Per-sample QC runner
# ---------------------------------------------------------------------
def process_qc_sample(sample: str) -> dict:
    paths = sample_paths(sample)
    input_path = locate_proseg_cdata(sample)
    metadata = SAMPLE_INFO[sample]

    print()
    print('=' * 90)
    print('QC sample:', sample)
    print('Input:', input_path)
    print('Annotated output:', paths['annotated'])
    print('Filtered output:', paths['filtered'])

    if (
        USE_EXISTING_OUTPUTS
        and not OVERWRITE_OUTPUTS
        and paths['annotated'].exists()
        and paths['filtered'].exists()
        and paths['summary'].exists()
    ):
        summary = json.loads(paths['summary'].read_text(encoding='utf-8'))
        version_matches = summary.get('pipeline_version') == PIPELINE_VERSION
        cell_cycle_matches = (
            not RUN_CELL_CYCLE_SCORING
            or summary.get('cell_cycle_status') == 'completed'
        )
        if version_matches and cell_cycle_matches:
            print('Reusing existing version-matched QC outputs.')
            return summary

        print(
            'Existing QC outputs are from an older configuration or lack '
            'completed cell-cycle scores; recomputing this sample.'
        )

    started = time.time()
    adata = ad.read_h5ad(input_path)
    adata.var_names_make_unique()
    maximum_fractional_error = ensure_integer_csr(adata)

    for key, value in metadata.items():
        adata.obs[key] = pd.Categorical([value] * adata.n_obs)
    adata.obs['sample'] = pd.Categorical([sample] * adata.n_obs)

    annotate_feature_classes(adata)
    calculate_qc_annotations(adata)
    mt_column, genes_column = add_cell_filter_flags(adata)

    adata.uns['proseg_resolvi_qc'] = {
        'pipeline_version': PIPELINE_VERSION,
        'sample': sample,
        **metadata,
        'source_proseg_cdata': str(input_path),
        'mt_high_threshold_percent': float(MT_HIGH_PERCENT),
        'min_genes_per_cell': int(MIN_GENES_PER_CELL),
        'min_cells_per_gene': int(MIN_CELLS_PER_GENE),
        'cell_filter_mt_column': mt_column,
        'cell_filter_genes_column': genes_column,
        'remove_deprecated_from_filtered': bool(REMOVE_DEPRECATED_FROM_FILTERED),
        'maximum_fractional_count_error': maximum_fractional_error,
        'count_matrix_location': 'X',
    }

    save_qc_figures(adata, sample, paths['report_dir'])

    filtered = adata[adata.obs['qc_pass'].to_numpy(dtype=bool)].copy()
    if REMOVE_DEPRECATED_FROM_FILTERED:
        filtered = filtered[:, ~filtered.var['is_deprecated'].to_numpy(dtype=bool)].copy()

    X = sp.csr_matrix(filtered.X)
    X.sum_duplicates()
    X.eliminate_zeros()
    n_cells_by_gene = np.bincount(X.indices, minlength=X.shape[1])
    gene_keep = n_cells_by_gene >= MIN_CELLS_PER_GENE
    filtered = filtered[:, gene_keep].copy()
    ensure_integer_csr(filtered)

    Xf = sp.csr_matrix(filtered.X)
    filtered.var['postfilter_total_counts'] = csr_col_sums(Xf)
    filtered.var['postfilter_n_cells_by_counts'] = np.bincount(
        Xf.indices, minlength=Xf.shape[1]
    ).astype(np.int64)

    cell_cycle_diagnostics = score_cell_cycle_on_qc_passed(filtered)
    propagate_cell_cycle_to_annotated(adata, filtered)
    save_cell_cycle_figures(filtered, sample, paths['report_dir'])

    adata.uns['proseg_resolvi_qc']['cell_cycle'] = cell_cycle_diagnostics
    filtered.uns['proseg_resolvi_qc'] = {
        **adata.uns['proseg_resolvi_qc'],
        'n_cells_before': int(adata.n_obs),
        'n_cells_after': int(filtered.n_obs),
        'n_genes_before': int(adata.n_vars),
        'n_genes_after': int(filtered.n_vars),
        'cell_cycle': cell_cycle_diagnostics,
    }

    # Both objects retain raw integer X. The normalized/log-transformed
    # temporary scoring matrix has already been deleted.
    ensure_integer_csr(adata)
    ensure_integer_csr(filtered)

    atomic_write_h5ad(adata, paths['annotated'])
    print('Saved QC-annotated object:', paths['annotated'])

    atomic_write_h5ad(filtered, paths['filtered'])
    print('Saved ResolVI-ready filtered object:', paths['filtered'])

    phase_counts = {
        str(key): int(value)
        for key, value in filtered.obs['phase'].value_counts().items()
    }

    summary = {
        'pipeline_version': PIPELINE_VERSION,
        'sample': sample,
        **metadata,
        'source': str(input_path),
        'annotated_h5ad': str(paths['annotated']),
        'filtered_h5ad': str(paths['filtered']),
        'n_cells_before': int(adata.n_obs),
        'n_cells_qc_pass': int(filtered.n_obs),
        'n_cells_mt_high': int(adata.obs['mt_high'].sum()),
        'n_cells_low_genes': int(adata.obs['low_genes'].sum()),
        'n_genes_before': int(adata.n_vars),
        'n_genes_after': int(filtered.n_vars),
        'n_deprecated_features': int(adata.var['is_deprecated'].sum()),
        'n_mt_features': int(adata.var['mt'].sum()),
        'n_ribo_features': int(adata.var['ribo'].sum()),
        'n_hb_features': int(adata.var['hb'].sum()),
        'median_nodep_n_genes': float(adata.obs['nodep_n_genes_by_counts'].median()),
        'median_nodep_total_counts': float(adata.obs['nodep_total_counts'].median()),
        'median_nodep_pct_counts_mt': float(adata.obs['nodep_pct_counts_mt'].median()),
        'cell_cycle_status': cell_cycle_diagnostics.get('status'),
        'cell_cycle_backend': cell_cycle_diagnostics.get('backend_used'),
        'cell_cycle_n_cells_scored': int(
            cell_cycle_diagnostics.get('n_cells_scored', 0)
        ),
        'cell_cycle_phase_counts': phase_counts,
        'cell_cycle_s_genes_present': int(
            len(cell_cycle_diagnostics.get('s_genes_present', []))
        ),
        'cell_cycle_g2m_genes_present': int(
            len(cell_cycle_diagnostics.get('g2m_genes_present', []))
        ),
        'runtime_minutes': (time.time() - started) / 60.0,
    }
    write_json(summary, paths['summary'])

    del adata, filtered
    gc.collect()
    return summary


In [6]:
# ---------------------------------------------------------------------
# Write shared configuration and run all selected samples
# ---------------------------------------------------------------------
(CONFIG_ROOT / 'signatures_v1.json').write_text(
    json.dumps(DEFAULT_SIGNATURES, indent=2), encoding='utf-8'
)
(CONFIG_ROOT / 'cell_cycle_genes_v1.json').write_text(
    json.dumps(
        {
            'S_phase': S_PHASE_GENES,
            'G2M_phase': G2M_PHASE_GENES,
            'target_sum': CELL_CYCLE_TARGET_SUM,
            'ctrl_as_ref': CELL_CYCLE_CTRL_AS_REF,
        },
        indent=2,
    ),
    encoding='utf-8',
)
pd.DataFrame.from_dict(SAMPLE_INFO, orient='index').rename_axis('sample').to_csv(
    CONFIG_ROOT / 'sample_manifest.csv'
)

qc_results = {}
qc_failures = {}
for sample in SECTION_NAMES:
    try:
        qc_results[sample] = process_qc_sample(sample)
    except Exception as exc:
        qc_failures[sample] = repr(exc)
        print(f'[FAILED] {sample}: {type(exc).__name__}: {exc}')
        if not CONTINUE_ON_ERROR:
            raise
    finally:
        gc.collect()

summary_table = pd.DataFrame.from_dict(qc_results, orient='index')
summary_table.to_csv(QC_REPORT_ROOT / 'all_samples_qc_summary.csv', index=False)
write_json(qc_failures, QC_REPORT_ROOT / 'all_samples_qc_failures.json')

print()
print('Completed samples:', sorted(qc_results))
print('Failures:', json.dumps(qc_failures, indent=2))
summary_table



QC sample: Screen_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_shared_qcprior_v1/Screen_39_21/Screen_39_21_proseg_sharedprior_cdata.h5ad
Annotated output: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_annotated/Screen_39_21/Screen_39_21_proseg_qc_annotated.h5ad
Filtered output: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_filtered/Screen_39_21/Screen_39_21_proseg_qc_filtered.h5ad


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cell-cycle scoring backend: Scanpy CPU
Saved QC-annotated object: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_annotated/Screen_39_21/Screen_39_21_proseg_qc_annotated.h5ad
Saved ResolVI-ready filtered object: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_filtered/Screen_39_21/Screen_39_21_proseg_qc_filtered.h5ad

QC sample: C2D15_39_21
Input: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_shared_qcprior_v1/C2D15_39_21/C2D15_39_21_proseg_sharedprior_cdata.h5ad
Annotated output: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_annotated/C2D15_39_21/C2D15_39_21_proseg_qc_annotated.h5ad
Filtered output: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/01_qc_filtered/C2D15_39_21/C2D15_39_21_proseg_qc_filtered.h5ad
Cell-cycle scoring back

,pipeline_version,sample,patient,cancer_type,biopsy_stage,source,annotated_h5ad,filtered_h5ad,n_cells_before,n_cells_qc_pass,...,median_nodep_n_genes,median_nodep_total_counts,median_nodep_pct_counts_mt,cell_cycle_status,cell_cycle_backend,cell_cycle_n_cells_scored,cell_cycle_phase_counts,cell_cycle_s_genes_present,cell_cycle_g2m_genes_present,runtime_minutes
Screen_39_21,2026-07-28-cellcycle-v2,Screen_39_21,patient_39_21,NSCLC,Screen,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,53351,22950,...,3.0,4.0,0.000000,completed,scanpy_cpu,22950,"{'G1': 11512, 'S': 6259, 'G2M': 5179, 'not_sco...",25,24,0.434023
C2D15_39_21,2026-07-28-cellcycle-v2,C2D15_39_21,patient_39_21,NSCLC,C2D15,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,14059,10822,...,139.0,198.0,2.028398,completed,scanpy_cpu,10822,"{'G1': 5040, 'S': 3596, 'G2M': 2186, 'not_scor...",24,24,0.073308
Screen_17_26,2026-07-28-cellcycle-v2,Screen_17_26,patient_17_26,NSCLC,Screen,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,58621,47896,...,407.0,530.0,5.259609,completed,scanpy_cpu,47896,"{'G1': 22496, 'S': 15311, 'G2M': 10089, 'not_s...",25,24,0.283534
C2D15_17_26,2026-07-28-cellcycle-v2,C2D15_17_26,patient_17_26,NSCLC,C2D15,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,97872,87913,...,569.0,740.0,4.347826,completed,rapids_singlecell,87913,"{'G1': 43184, 'S': 24880, 'G2M': 19849, 'not_s...",25,24,0.483791
Screen_18_23,2026-07-28-cellcycle-v2,Screen_18_23,patient_18_23,melanoma,Screen,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,83114,72386,...,157.0,225.0,2.353249,completed,rapids_singlecell,72386,"{'S': 39331, 'G1': 19191, 'G2M': 13864, 'not_s...",25,24,0.188395
C2D15_18_23,2026-07-28-cellcycle-v2,C2D15_18_23,patient_18_23,melanoma,C2D15,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,20043,18594,...,422.0,692.0,3.189834,completed,scanpy_cpu,18594,"{'G1': 7996, 'S': 6976, 'G2M': 3622, 'not_scor...",25,24,0.107731
Screen_16_22,2026-07-28-cellcycle-v2,Screen_16_22,patient_16_22,melanoma,Screen,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,106351,70438,...,52.0,58.0,2.105263,completed,rapids_singlecell,70438,"{'G1': 27268, 'S': 25593, 'G2M': 17577, 'not_s...",25,24,0.156196
C2D15_16_22,2026-07-28-cellcycle-v2,C2D15_16_22,patient_16_22,melanoma,C2D15,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,81053,76633,...,437.0,517.0,3.549439,completed,rapids_singlecell,76633,"{'G1': 39759, 'G2M': 19123, 'S': 17751, 'not_s...",25,24,0.288666
Screen_30_16,2026-07-28-cellcycle-v2,Screen_30_16,patient_30_16,melanoma,Screen,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,162836,66977,...,2.0,2.0,0.000000,completed,rapids_singlecell,66977,"{'G1': 29842, 'S': 20286, 'G2M': 16849, 'not_s...",25,24,0.284631
C2D15_30_16,2026-07-28-cellcycle-v2,C2D15_30_16,patient_30_16,melanoma,C2D15,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,637581,351799,...,32.0,36.0,10.502283,completed,rapids_singlecell,351799,"{'G2M': 135397, 'G1': 115377, 'S': 101025, 'no...",25,24,0.391285


In [7]:
#also record environment location
import sys
print(sys.executable)

/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/bin/python
